### Connect postgresql database

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# 数据库配置
username = "XXXXXX"
password = "YYYYYY"
host = "localhost"
port = 5432
database = "eyewear-data"

# 创建连接
engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

# 查询数据
sql = """
SELECT
    o.order_id,
    o.customer_id,
    o.order_date,
    o.order_status,
    o.total_price_before_tax,
    oi.product_id,
    oi.quantity,
    oi.product_name,
    oi.unit_price,
    oi.line_price_before_tax,
    oi.is_free_gift,
    pa.campaign_id
FROM "Order" o
JOIN "OrderItem" oi
    ON o.order_id = oi.order_id
JOIN "PromotionActivity" pa
    ON o.campaign_id = pa.campaign_id
WHERE o.order_status IN ('Completed', 'Shipped')
"""

df_order_completed_shipped = pd.read_sql(sql, engine)

# 查看数据
df_order_completed_shipped

### Debug: check out df columns name

In [ ]:
df_order_completed_shipped.columns

### Statistics on sales qty for 2023-2024 each month

In [ ]:
df_order_completed_shipped['order_date'] = pd.to_datetime(df_order_completed_shipped['order_date'])
df_order_completed_shipped['order_month'] = df_order_completed_shipped['order_date'].dt.to_period('M')

# 如果要排除赠品
df_valid = df_order_completed_shipped[
    df_order_completed_shipped['is_free_gift'] != True
]

df_monthly_qty = (
    df_valid.groupby('order_month', as_index=False)['quantity']
      .sum()
      .sort_values('order_month')
)

df_monthly_qty['order_month'] = df_monthly_qty['order_month'].dt.to_timestamp()

df_monthly_qty


### Statistics on sales qty for 2023-2024 each quarter

In [ ]:
df_order_completed_shipped['order_date'] = pd.to_datetime(df_order_completed_shipped['order_date'])
df_order_completed_shipped['order_quarter'] = df_order_completed_shipped['order_date'].dt.to_period('Q')

# 排除赠品
df_valid = df_order_completed_shipped[df_order_completed_shipped['is_free_gift'] != True]

df_quarterly_qty = (
    df_valid.groupby('order_quarter', as_index=False)['quantity']
      .sum()
      .sort_values('order_quarter')
)

df_quarterly_qty['order_quarter'] = df_quarterly_qty['order_quarter'].astype(str).str.replace('Q', '-Q')
df_quarterly_qty


### Statistics on sales qty for 2023-2024 each year

In [ ]:
df_order_completed_shipped['order_year'] = df_order_completed_shipped['order_date'].dt.year

# 排除赠品
df_valid = df_order_completed_shipped[df_order_completed_shipped['is_free_gift'] != True]

df_annual_qty = (
    df_valid.groupby('order_year', as_index=False)['quantity']
      .sum()
      .sort_values('order_year')
)

df_annual_qty

In [ ]:
# 关闭数据库连接
engine.dispose()